In [55]:
"""
Scooter Parking Photo Classifier - Prompt Optimization Loop

This notebook demonstrates how to use the prompt optimization system
to iteratively improve prompts for classifying scooter parking photos.
"""

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import importlib

# Import the optimization module
sys.path.append(str(Path.cwd()))

# Reload module to pick up latest changes (useful during development)
import prompt_optimization_classifier
importlib.reload(prompt_optimization_classifier)

from prompt_optimization_classifier import (
    PromptOptimizer, Config, LLMClient, Metrics
)

print("✓ Imports successful")
print("✓ Module reloaded (ensures latest code changes are used)")


✓ Imports successful
✓ Module reloaded (ensures latest code changes are used)


In [56]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Configure the optimization parameters
# Temperature is fixed at 0.2 - only prompts will be optimized
FIXED_TEMPERATURE = 0.2

config = Config(
    temperatures=[FIXED_TEMPERATURE],  # Fixed temperature - not optimized
    precision_threshold=0.90,           # Minimum precision for FAIL class
    max_iterations=5,                   # Maximum optimization iterations
    fail_keyword="expected_fail",       # Filename keyword for FAIL label
    pass_keyword="expected_pass",       # Filename keyword for PASS label
    dev_split=0.7,                      # 70% dev, 30% test
    random_seed=42                      # For reproducible splits
)

print("Configuration:")
print(f"  Temperature: {FIXED_TEMPERATURE} (FIXED - only prompts optimized)")
print(f"  Precision threshold: {config.precision_threshold}")
print(f"  Max iterations: {config.max_iterations}")
print(f"  Dev/Test split: {config.dev_split:.0%}/{1-config.dev_split:.0%}")


Configuration:
  Temperature: 0.2 (FIXED - only prompts optimized)
  Precision threshold: 0.9
  Max iterations: 5
  Dev/Test split: 70%/30%


In [57]:
# ============================================================================
# INITIAL PROMPTS (CUSTOMIZE THESE)
# ============================================================================
# Define your initial system and user prompts here.
# These will be used as the starting point for optimization.

INITIAL_SYSTEM_PROMPT = """"
You are a micromobility parking enforcement officer.
Analyze this parking photo of a shared e-scooter or bicycle.
Rate it as PASS or FAIL and provide feedback accordingly.

Condense your feedback into a single, short sentence.
If FAIL, make it actionable so the rider knows what they need to improve.
"""

INITIAL_USER_PROMPT = """
Analyze this parking photo according to the rules below and decide whether the vehicle is correctly parked.

Vehicle types to evaluate

Only evaluate these four vehicle types used in this market:
• Bird scooters (white-and-blue)
• Spin scooters (orange)
• Bird e-bikes (blue frame with black rear fender)
• Bird e-bikes (white frame with blue branding)

1. Identify the corral

A micromobility parking corral is defined only by one or more of the following indicators:
• A painted white box on the ground
• A painted “P” symbol
• A painted scooter or bicycle icon
• Black-and-white posts or bollards arranged to outline a parking zone

Any of these alone is sufficient to confirm a corral.


3. Snow or partial visibility

• If a P symbol or scooter/bike icon is visible, treat that area as the corral even if boundary paint is partially missing or covered by snow.
• If a Bird or Spin vehicle is parked directly on top of, or centered over, a visible P symbol or scooter/bike icon, and is upright, treat it as correctly parked inside the corral (PASS) even if snow hides the rest of the markings.
• When black-and-white posts or bollards form a corral, only the area inside those posts counts as the corral. Any Bird or Spin vehicle outside the posts is outside the corral, even if it is close to a P symbol.

4. Parking correctness

• Consider all Bird and Spin vehicles visible in the photo. If any of them is outside the corral → FAIL.
• If a vehicle is partly inside the corral, the majority of the vehicle must be inside to PASS.
• The vehicle must be upright, fully visible, and not held by a person.

"""

print("✓ Initial prompts defined")
print(f"\nSystem prompt length: {len(INITIAL_SYSTEM_PROMPT)} characters")
print(f"User prompt length: {len(INITIAL_USER_PROMPT)} characters")


✓ Initial prompts defined

System prompt length: 303 characters
User prompt length: 1610 characters


In [58]:
# ============================================================================
# ANTHROPIC CLAUDE LLM CLIENT
# ============================================================================

class AnthropicLLMClient(LLMClient):
    """Anthropic Claude client for vision and text tasks"""
    
    def __init__(self, api_key: str, model: str = "claude-3-5-sonnet-20241022"):
        """
        Initialize Anthropic client.
        
        Args:
            api_key: Anthropic API key
            model: Claude model name (default: claude-3-5-sonnet-20241022, latest available)
                   Note: Claude 4.0 is not yet released. Using Claude 3.5 Sonnet (latest).
        """
        import anthropic
        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = model
    
    def call_llm(
        self,
        system_prompt: str,
        user_prompt: str,
        temperature: float,
        model: str = None,
        image_path: str = None
    ) -> str:
        """Call Claude for text-only tasks"""
        # Ensure user_prompt is never None - convert to empty string if needed
        if user_prompt is None:
            user_prompt = ""
        if system_prompt is None:
            system_prompt = ""
        if temperature is None:
            temperature = 0.1
        
        message = self.client.messages.create(
            model=model or self.model,
            max_tokens=1024,
            temperature=float(temperature),
            system=system_prompt,
            messages=[
                {"role": "user", "content": user_prompt}
            ]
        )
        
        return message.content[0].text
    
    def call_vision_llm(
        self,
        system_prompt: str,
        user_prompt: str,
        image_path: str,
        temperature: float,
        model: str = None
    ) -> str:
        """Call Claude for vision tasks (image analysis)"""
        from pathlib import Path
        import base64
        import mimetypes
        
        # Ensure user_prompt is never None - convert to empty string if needed
        if user_prompt is None:
            user_prompt = ""
        
        # Ensure temperature is a float
        if temperature is None:
            temperature = 0.1  # Default temperature
        
        # Prepare image - Anthropic requires local files
        if not Path(image_path).exists():
            raise ValueError(f"Image file not found: {image_path}. Anthropic requires local image files.")
        
        # Read and encode image
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode('utf-8')
        
        # Determine media type
        media_type, _ = mimetypes.guess_type(image_path)
        if not media_type:
            # Default to jpeg if can't determine
            media_type = "image/jpeg"
        
        # Build content array with text first, then image
        # Anthropic format: content is a list with text dict and image dict
        content = []
        
        # Add text prompt if not empty
        if user_prompt.strip():
            content.append({"type": "text", "text": user_prompt})
        
        # Add image
        content.append({
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": media_type,
                "data": image_data
            }
        })
        
        message = self.client.messages.create(
            model=model or self.model,
            max_tokens=1024,
            temperature=float(temperature),
            system=system_prompt if system_prompt else "",
            messages=[
                {
                    "role": "user",
                    "content": content
                }
            ]
        )
        
        return message.content[0].text


print("✓ Anthropic Claude LLM client defined")
print("\nUsing Claude 3.5 Sonnet (latest available model)")
print("Note: Claude 4.0 is not yet released. Update model name when available.")


✓ Anthropic Claude LLM client defined

Using Claude 3.5 Sonnet (latest available model)
Note: Claude 4.0 is not yet released. Update model name when available.


In [59]:
# ============================================================================
# INITIALIZE ANTHROPIC CLIENT
# ============================================================================

import os
from dotenv import load_dotenv

load_dotenv()

# Get Anthropic API key from environment
API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found in environment. "
        "Please set it in your .env file or environment variables."
    )

# Initialize Anthropic Claude client
# Using Claude 3.5 Sonnet (latest available)
# Update model name when Claude 4.0 is released
llm_client = AnthropicLLMClient(
    api_key=API_KEY,
    model="claude-sonnet-4-20250514"  # Latest Claude model
)

print(f"✓ Initialized Anthropic Claude client")
print(f"  Model: claude-sonnet-4-20250514")


✓ Initialized Anthropic Claude client
  Model: claude-sonnet-4-20250514


In [60]:
# ============================================================================
# LOAD DATA
# ============================================================================

# Check if required variables are defined
if 'config' not in locals():
    raise NameError("Variable 'config' is not defined. Please run Cell 1 (CONFIGURATION) first.")
if 'llm_client' not in locals():
    raise NameError("Variable 'llm_client' is not defined. Please run Cell 4 (INITIALIZE ANTHROPIC CLIENT) first.")
if 'INITIAL_SYSTEM_PROMPT' not in locals() or 'INITIAL_USER_PROMPT' not in locals():
    raise NameError("Variables 'INITIAL_SYSTEM_PROMPT' and 'INITIAL_USER_PROMPT' are not defined. Please run Cell 2 (INITIAL PROMPTS) first.")

# Path to CSV file with Denver nests photos
CSV_PATH = "Denver AI photo review demo/denver_nests_photos.csv"

# Initialize optimizer with custom initial prompts (defined in Cell 2)
optimizer = PromptOptimizer(
    config, 
    llm_client,
    initial_system_prompt=INITIAL_SYSTEM_PROMPT,
    initial_user_prompt=INITIAL_USER_PROMPT
)

# Load and prepare data
df = optimizer.load_and_prepare_data(CSV_PATH)

# Display data summary
print(f"\nData loaded: {len(df)} images")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nSample rows:")
print(df.head())


2025-12-08 00:57:41,899 - INFO - Loading data from Denver AI photo review demo/denver_nests_photos.csv
2025-12-08 00:57:41,905 - INFO - Loaded 20 images: {'FAIL': 15, 'PASS': 5}



Data loaded: 20 images

Label distribution:
label
FAIL    15
PASS     5
Name: count, dtype: int64

Sample rows:
                                          image_path label
0  /Users/assafkazakov/projects/playground/Denver...  FAIL
1  /Users/assafkazakov/projects/playground/Denver...  FAIL
2  /Users/assafkazakov/projects/playground/Denver...  FAIL
3  /Users/assafkazakov/projects/playground/Denver...  FAIL
4  /Users/assafkazakov/projects/playground/Denver...  FAIL


In [61]:
# ============================================================================
# SPLIT DATA
# ============================================================================

# Check if df exists (from previous cell)
if 'df' not in locals():
    raise NameError(
        "Variable 'df' is not defined. Please run Cell 5 (LOAD DATA) first. "
        "Make sure all previous cells have been executed successfully."
    )

optimizer.split_data(df)

print("✓ Data split complete")
print(f"\nDev set: {len(optimizer.dev_data)} images")
print(f"Test set: {len(optimizer.test_data)} images")


2025-12-08 00:57:41,909 - INFO - Split: 14 dev, 6 test
2025-12-08 00:57:41,910 - INFO - Dev labels: {'FAIL': 9, 'PASS': 5}
2025-12-08 00:57:41,910 - INFO - Test labels: {'FAIL': 6}


✓ Data split complete

Dev set: 14 images
Test set: 6 images


In [62]:
# ============================================================================
# RUN OPTIMIZATION LOOP
# ============================================================================

optimizer.run_optimization_loop()

print("\n✓ Optimization complete!")
print(f"\nBest configuration:")
print(f"  Temperature: {optimizer.best_temperature}")
print(f"  Metrics: {optimizer.best_metrics}")


2025-12-08 00:57:41,913 - INFO - ================================================================================
2025-12-08 00:57:41,913 - INFO - Starting Prompt Optimization Loop
2025-12-08 00:57:41,913 - INFO - ================================================================================
2025-12-08 00:57:41,913 - INFO - 
2025-12-08 00:57:41,913 - INFO - Iteration 0 - Prompt Optimization Only
2025-12-08 00:57:41,914 - INFO - ================================================================================
2025-12-08 00:57:41,914 - INFO - Temperature: 0.2 (fixed - not optimized)
2025-12-08 00:57:41,914 - INFO - Evaluating on 14 images (temp=0.2)...
2025-12-08 00:57:41,914 - INFO - System Prompt: "
You are a micromobility parking enforcement officer.
Analyze this parking photo of a shared e-scoo...
2025-12-08 00:57:41,914 - INFO - User Prompt: 
Analyze this parking photo according to the rules below and decide whether the vehicle is correctly...
2025-12-08 00:57:48,540 - INFO - HTTP 


✓ Optimization complete!

Best configuration:
  Temperature: 0.2
  Metrics: Metrics (temp=0.2, v=0): Recall=1.000, Precision=0.643, F2=0.900, TP=9, FP=5, FN=0, TN=0


In [63]:
# ============================================================================
# FINAL EVALUATION ON TEST SET
# ============================================================================

test_metrics, test_misclassified = optimizer.evaluate_on_test_set()

print(f"\n✓ Final evaluation complete!")
print(f"\nTest set performance:")
print(f"  Recall (FAIL): {test_metrics.recall_fail:.3f}")
print(f"  Precision (FAIL): {test_metrics.precision_fail:.3f}")
print(f"  F2 Score: {test_metrics.f2_fail:.3f}")
print(f"\nConfusion Matrix (FAIL=positive):")
print(f"  TP={test_metrics.tp}, FP={test_metrics.fp}")
print(f"  FN={test_metrics.fn}, TN={test_metrics.tn}")


2025-12-08 01:02:44,366 - INFO - 
2025-12-08 01:02:44,367 - INFO - Final Evaluation on Test Set
2025-12-08 01:02:44,367 - INFO - ================================================================================
2025-12-08 01:02:44,368 - INFO - Using configuration:
2025-12-08 01:02:44,368 - INFO -   Temperature: 0.2 (type: float)
2025-12-08 01:02:44,368 - INFO -   System Prompt length: 303 chars
2025-12-08 01:02:44,368 - INFO -   User Prompt length: 1610 chars
2025-12-08 01:02:44,368 - INFO - Evaluating on 6 images (temp=0.2)...
2025-12-08 01:02:44,369 - INFO - System Prompt: "
You are a micromobility parking enforcement officer.
Analyze this parking photo of a shared e-scoo...
2025-12-08 01:02:44,369 - INFO - User Prompt: 
Analyze this parking photo according to the rules below and decide whether the vehicle is correctly...
2025-12-08 01:02:49,614 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-12-08 01:02:49,617 - WARNING - JSON parse error for 


✓ Final evaluation complete!

Test set performance:
  Recall (FAIL): 1.000
  Precision (FAIL): 1.000
  F2 Score: 1.000

Confusion Matrix (FAIL=positive):
  TP=6, FP=0
  FN=0, TN=0


In [64]:
# ============================================================================
# VIEW FINAL PROMPTS
# ============================================================================

print("=" * 80)
print("FINAL OPTIMIZED PROMPTS")
print("=" * 80)

print("\nSYSTEM PROMPT:")
print("-" * 80)
print(optimizer.best_prompt_system)

print("\n\nUSER PROMPT:")
print("-" * 80)
print(optimizer.best_prompt_user)

print("\n\nTEMPERATURE:")
print("-" * 80)
print(f"{optimizer.best_temperature}")


FINAL OPTIMIZED PROMPTS

SYSTEM PROMPT:
--------------------------------------------------------------------------------
"
You are a micromobility parking enforcement officer.
Analyze this parking photo of a shared e-scooter or bicycle.
Rate it as PASS or FAIL and provide feedback accordingly.

Condense your feedback into a single, short sentence.
If FAIL, make it actionable so the rider knows what they need to improve.



USER PROMPT:
--------------------------------------------------------------------------------

Analyze this parking photo according to the rules below and decide whether the vehicle is correctly parked.

Vehicle types to evaluate

Only evaluate these four vehicle types used in this market:
• Bird scooters (white-and-blue)
• Spin scooters (orange)
• Bird e-bikes (blue frame with black rear fender)
• Bird e-bikes (white frame with blue branding)

1. Identify the corral

A micromobility parking corral is defined only by one or more of the following indicators:
• A paint

In [65]:
# ============================================================================
# ANALYZE MISCLASSIFICATIONS
# ============================================================================

if test_misclassified:
    print(f"\nMisclassified examples on test set: {len(test_misclassified)}")
    
    # Group by error type
    false_positives = [m for m in test_misclassified if m['ground_truth'] == 'PASS' and m['model_decision'] == 'FAIL']
    false_negatives = [m for m in test_misclassified if m['ground_truth'] == 'FAIL' and m['model_decision'] == 'PASS']
    
    print(f"\nFalse Positives (PASS labeled as FAIL): {len(false_positives)}")
    print(f"False Negatives (FAIL labeled as PASS): {len(false_negatives)}")
    
    # Show sample errors
    if false_negatives:
        print("\nSample False Negatives (most critical - missed FAIL cases):")
        for ex in false_negatives[:5]:
            print(f"  - {ex['image_path']}")
            print(f"    Model reason: {ex.get('reason', 'N/A')[:100]}")
    
    if false_positives:
        print("\nSample False Positives:")
        for ex in false_positives[:5]:
            print(f"  - {ex['image_path']}")
            print(f"    Model reason: {ex.get('reason', 'N/A')[:100]}")
else:
    print("✓ No misclassifications on test set!")


✓ No misclassifications on test set!
